# 🚀 OmniRoute AI Gateway — Google Colab Worker (for Humanizer Pro & VideoStudio)

Runs the [OmniRoute AI gateway](https://github.com/cheaper-inference/OmniRoute) (`v3.8.53` on Node.js `22.22.2`), automatically restores your saved providers & API keys from Google Drive (`/content/drive/MyDrive/OmniRoute/omniroute-backup.tar.gz`), and optionally opens a **Cloudflare Tunnel (`*.trycloudflare.com`)** so your desktop apps (**Humanizer Pro**, **VideoStudio**, Cursor, etc.) can connect to it remotely.

### Quick Start
1. **Runtime → Change runtime type → Python 3 → CPU** (no GPU needed for API routing).
2. Run **Step 1** (mounts Google Drive & restores your saved OmniRoute database and keys).
3. Run **Step 2** (installs Node.js 22 & OmniRoute 3.8.53, including postinstall runtime packages).
4. Run **Step 3** (starts OmniRoute; only asks for an initial password if no saved settings exist).
5. Run **Step 4** to open the **Dashboard** in Colab or start a **Public Tunnel (`*.trycloudflare.com`)** for your local desktop `.env`.
6. When finished, run **Step 6 (Save & Stop)** to back up any dashboard changes back to Google Drive.

## 1. Mount Google Drive & Restore Saved Settings
Restores `/content/omniroute-data` from `/content/drive/MyDrive/OmniRoute/omniroute-backup.tar.gz` if present.

In [ ]:
import shutil
import tarfile
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

SAVE_DIR = Path("/content/drive/MyDrive/OmniRoute")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

BACKUP = SAVE_DIR / "omniroute-backup.tar.gz"
DATA = Path("/content/omniroute-data")

def _has_saved_db(data_dir: Path) -> bool:
    db = data_dir / "storage.sqlite"
    return db.is_file() and db.stat().st_size > 0

if BACKUP.exists() and not _has_saved_db(DATA):
    if DATA.exists():
        shutil.rmtree(DATA, ignore_errors=True)
    with tarfile.open(BACKUP, "r:gz") as archive:
        for member in archive.getmembers():
            path = Path(member.name)
            if (
                path.is_absolute()
                or ".." in path.parts
                or not path.parts
                or path.parts[0] != "omniroute-data"
                or not (member.isfile() or member.isdir())
            ):
                raise RuntimeError(f"Unexpected backup entry: {member.name}")
        archive.extractall("/content", filter="data")
    print(f"✅ Settings restored from Google Drive ({BACKUP}).")
elif _has_saved_db(DATA):
    print("✅ Drive connected. Existing /content/omniroute-data settings are active.")
else:
    print("ℹ️ Drive connected. No backup archive found yet; a fresh setup will be created in Step 3.")

## 2. Install Node.js 22.22.2 & OmniRoute 3.8.53
Installs Node.js and OmniRoute locally under `/content/omniroute-runtime` and prepares the standalone runtime bundles (`@swc/helpers`, `sql.js`, `node-machine-id`).

In [ ]:
import hashlib
import os
import pathlib
import platform
import subprocess
import tarfile
import urllib.request

if not pathlib.Path("/content").exists():
    raise RuntimeError("Run this notebook in Google Colab with a Python 3 runtime.")
if platform.machine() != "x86_64":
    raise RuntimeError("This installer expects the standard Colab x86_64 Linux runtime.")

NODE_VERSION = "22.22.2"
OMNIROUTE_VERSION = "3.8.53"
root = pathlib.Path("/content/omniroute-runtime")
root.mkdir(exist_ok=True)
node_dir = root / f"node-v{NODE_VERSION}-linux-x64"
archive_name = node_dir.name + ".tar.xz"
base = f"https://nodejs.org/dist/v{NODE_VERSION}/"

if not (node_dir / "bin/node").exists():
    print(f"⏳ Downloading Node.js v{NODE_VERSION}...")
    archive = root / archive_name
    urllib.request.urlretrieve(base + archive_name, archive)
    checksums = urllib.request.urlopen(base + "SHASUMS256.txt").read().decode()
    expected = next(
        line.split()[0] for line in checksums.splitlines()
        if line.split()[-1] == archive_name
    )
    actual = hashlib.sha256(archive.read_bytes()).hexdigest()
    if actual != expected:
        raise RuntimeError("Node.js download checksum mismatch.")
    with tarfile.open(archive, "r:xz") as bundle:
        bundle.extractall(root, filter="data")
    archive.unlink()

local_bin = root / "node_modules" / ".bin"
os.environ["PATH"] = str(local_bin) + os.pathsep + str(node_dir / "bin") + os.pathsep + os.environ.get("PATH", "")

omniroute_pkg = root / "node_modules" / "omniroute"
if not (local_bin / "omniroute").exists():
    print(f"⏳ Installing omniroute@{OMNIROUTE_VERSION} (takes ~1-2 minutes)...")
    subprocess.run(["node", "--version"], check=True)
    subprocess.run(
        ["npm", "install", "--no-audit", "--no-fund", f"omniroute@{OMNIROUTE_VERSION}", "@swc/helpers@0.5.19"],
        cwd=str(root),
        check=True,
    )
    postinstall = omniroute_pkg / "scripts" / "build" / "postinstall.mjs"
    if postinstall.exists():
        subprocess.run(["node", str(postinstall)], cwd=str(omniroute_pkg), check=True)

print("✅ OmniRoute installation completed.")

## 3. Start OmniRoute Server
Starts OmniRoute on port `20128`. If your database was restored from Google Drive, it uses your saved password and encryption key automatically without prompting.

In [ ]:
import getpass
import os
import pathlib
import secrets
import sqlite3
import subprocess
import time
import urllib.error
import urllib.request

PORT = 20128
DATA_DIR = pathlib.Path("/content/omniroute-data")
DATA_DIR.mkdir(mode=0o700, exist_ok=True)
LOG_PATH = pathlib.Path("/content/omniroute.log")

root = pathlib.Path("/content/omniroute-runtime")
node_dir = root / "node-v22.22.2-linux-x64"
local_bin = root / "node_modules" / ".bin"
os.environ["PATH"] = str(local_bin) + os.pathsep + str(node_dir / "bin") + os.pathsep + os.environ.get("PATH", "")

def _is_server_up(port: int = PORT) -> bool:
    try:
        with urllib.request.urlopen(f"http://127.0.0.1:{port}/", timeout=2) as resp:
            return resp.status == 200
    except urllib.error.HTTPError as err:
        return err.code in (401, 403)
    except Exception:
        return False

def _has_saved_password(data_dir: Path) -> bool:
    db = data_dir / "storage.sqlite"
    if not db.is_file():
        return False
    try:
        conn = sqlite3.connect(str(db))
        row = conn.execute(
            "SELECT value FROM key_value WHERE namespace = 'settings' AND key = 'password'"
        ).fetchone()
        conn.close()
        return bool(row and row[0])
    except Exception:
        return False

existing_proc = globals().get("omniroute_process")
if (existing_proc is not None and existing_proc.poll() is None) or _is_server_up(PORT):
    print(f"✅ OmniRoute is already running on port {PORT}; continue to Step 4.")
else:
    if _has_saved_password(DATA_DIR):
        password = "restored-from-drive-backup"
        print("🔑 Using saved dashboard password from restored database.")
    else:
        password = getpass.getpass("Choose initial dashboard password (at least 12 characters): ")
        if len(password) < 12:
            raise ValueError("Choose a password of at least 12 characters, then rerun this cell.")

    secret_path = DATA_DIR / ".colab-jwt-secret"
    if not secret_path.exists():
        secret_path.write_text(secrets.token_hex(32))
        secret_path.chmod(0o600)

    env_path = DATA_DIR / ".env"
    storage_key = ""
    if env_path.is_file():
        for line in env_path.read_text(encoding="utf-8").splitlines():
            if line.strip().startswith("STORAGE_ENCRYPTION_KEY="):
                storage_key = line.split("=", 1)[1].strip().strip("\"'")
                break
    if not storage_key:
        storage_key = secrets.token_hex(32)
        env_path.write_text(f"STORAGE_ENCRYPTION_KEY={storage_key}\n", encoding="utf-8")
        env_path.chmod(0o600)

    server_env = os.environ.copy()
    server_env.update({
        "DATA_DIR": str(DATA_DIR),
        "INITIAL_PASSWORD": password,
        "JWT_SECRET": secret_path.read_text().strip(),
        "STORAGE_ENCRYPTION_KEY": storage_key,
        "REQUIRE_API_KEY": "true",
        "HOSTNAME": "0.0.0.0",
        "PORT": str(PORT),
    })

    with LOG_PATH.open("w") as log:
        omniroute_bin = str(local_bin / "omniroute")
        omniroute_process = subprocess.Popen(
            [omniroute_bin, "--port", str(PORT), "--no-open"],
            env=server_env,
            stdout=log,
            stderr=subprocess.STDOUT,
            start_new_session=True,
        )
    del password, server_env

    ready = False
    for _ in range(90):
        if omniroute_process.poll() is not None:
            print(LOG_PATH.read_text() if LOG_PATH.exists() else "")
            raise RuntimeError("OmniRoute exited early. See log output above.")
        if _is_server_up(PORT):
            ready = True
            print(f"✅ OmniRoute is running on http://127.0.0.1:{PORT} (API: http://127.0.0.1:{PORT}/v1)")
            break
        time.sleep(1)
    if not ready:
        raise RuntimeError("Startup timed out. Run the log viewer cell below to inspect /content/omniroute.log.")

In [ ]:
# Optional: View /content/omniroute.log
import pathlib

log_file = pathlib.Path("/content/omniroute.log")
if log_file.exists():
    print(log_file.read_text())
else:
    print("Log file /content/omniroute.log does not exist yet. Run Step 3 first.")

## 4. Open Dashboard & Start Public Tunnel for Desktop Apps
- **Cell 4A** opens the OmniRoute Dashboard in your browser via Colab's secure port proxy.
- **Cell 4B** starts a free **Cloudflare Quick Tunnel (`https://....trycloudflare.com`)** and prints the exact `.env` lines (`OMNIROUTE_BASE_URL`, `OMNIROUTE_API_KEY`, `OMNIROUTE_MODELS`) to paste into your local **Humanizer Pro** `.env` on your Desktop.

In [ ]:
# 4A. Open OmniRoute Dashboard in your browser
from google.colab import output
from IPython.display import HTML, display

PORT = 20128
proxy_url = output.eval_js(f"google.colab.kernel.proxyPort({PORT})")

display(HTML(f"""
    <div style="margin: 15px 0; padding: 15px; border: 1px solid #ddd; border-radius: 8px; background-color: #f8f9fa;">
        <h3 style="margin-top: 0; color: #333;">OmniRoute Dashboard Ready</h3>
        <p style="color: #555;">Click the link below to open your dashboard in a new tab:</p>
        <a href="{proxy_url}" target="_blank" style="font-size: 18px; font-weight: bold; color: #1a73e8; text-decoration: none;">
            Launch Dashboard ↗️
        </a>
    </div>
"""))

In [ ]:
# 4B. Expose OmniRoute via Cloudflare Quick Tunnel for Desktop Humanizer Pro / VideoStudio
import os
import pathlib
import re
import stat
import subprocess
import time
import urllib.request

PORT = 20128
cf_bin = pathlib.Path("/content/cloudflared")
if not cf_bin.exists():
    print("⏳ Downloading cloudflared...")
    urllib.request.urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        cf_bin,
    )
    cf_bin.chmod(cf_bin.stat().st_mode | stat.S_IEXEC)

tunnel_log = pathlib.Path("/content/cloudflared.log")
existing_tunnel = globals().get("tunnel_process")
if existing_tunnel is not None and existing_tunnel.poll() is None:
    pass
else:
    with tunnel_log.open("w") as f:
        tunnel_process = subprocess.Popen(
            [str(cf_bin), "tunnel", "--url", f"http://127.0.0.1:{PORT}", "--no-autoupdate"],
            stdout=f,
            stderr=subprocess.STDOUT,
            start_new_session=True,
        )

public_url = None
for _ in range(30):
    if tunnel_log.exists():
        m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", tunnel_log.read_text())
        if m:
            public_url = m.group(0)
            break
    time.sleep(1)

if public_url:
    print("=" * 72)
    print(f"🌐 Public OmniRoute Gateway URL: {public_url}/v1")
    print("=" * 72)
    print("Paste these lines into your Desktop Humanizer .env file:\n")
    print(f"OMNIROUTE_BASE_URL={public_url}/v1")
    print("OMNIROUTE_API_KEY=<your-omniroute-api-key>")
    print("OMNIROUTE_MODELS=auto/best-free,groq/openai/gpt-oss-120b,groq/qwen/qwen3.8-27b,gemini/gemini-3.5-flash,gemini/gemini-3.1-flash-lite,openrouter/auto")
    print("=" * 72)
else:
    print("⚠️ Could not detect trycloudflare URL yet. Check /content/cloudflared.log:")
    print(tunnel_log.read_text() if tunnel_log.exists() else "")

## 5. Verify the Gateway (`/v1/models` & Live Completion Test)
Automatically reads your saved OmniRoute API key from `/content/omniroute-data` (or prompts if none is saved) and verifies both `/v1/models` and a live `/v1/chat/completions` call.

In [ ]:
import getpass
import json
import os
import pathlib
import subprocess
import urllib.error
import urllib.request

PORT = 20128
DATA_DIR = pathlib.Path("/content/omniroute-data")
root = pathlib.Path("/content/omniroute-runtime")
node_bin = root / "node-v22.22.2-linux-x64" / "bin" / "node"

def _load_saved_api_key() -> str:
    db = DATA_DIR / "storage.sqlite"
    env_file = DATA_DIR / ".env"
    if not (db.is_file() and env_file.is_file() and node_bin.exists()):
        return ""
    try:
        import sqlite3
        conn = sqlite3.connect(str(db))
        row = conn.execute("SELECT key FROM api_keys WHERE is_active = 1 LIMIT 1").fetchone()
        conn.close()
        if not row or not row[0]:
            return ""
        enc_key = row[0]
        if not enc_key.startswith("enc:v1:"):
            return enc_key
        js = (
            "const fs=require('fs'),crypto=require('crypto');"
            "const env=fs.readFileSync('/content/omniroute-data/.env','utf8');"
            "const sec=env.split('STORAGE_ENCRYPTION_KEY=')[1].trim().split('\\n')[0].trim();"
            "const k=crypto.scryptSync(sec,'omniroute-field-encryption-v1',32);"
            "const [iv,ct,tag]=process.argv[1].slice(7).split(':');"
            "const d=crypto.createDecipheriv('aes-256-gcm',k,Buffer.from(iv,'hex'),{authTagLength:16});"
            "d.setAuthTag(Buffer.from(tag,'hex'));"
            "process.stdout.write(d.update(ct,'hex','utf8')+d.final('utf8'));"
        )
        return subprocess.check_output([str(node_bin), "-e", js, enc_key]).decode().strip()
    except Exception:
        return ""

api_key = _load_saved_api_key() or getpass.getpass("OmniRoute API key: ")
try:
    req = urllib.request.Request(
        f"http://127.0.0.1:{PORT}/v1/models",
        headers={"Authorization": "Bearer " + api_key},
    )
    with urllib.request.urlopen(req, timeout=30) as response:
        result = json.loads(response.read().decode())
    models = result.get("data", [])
    print(f"✅ /v1/models responded with {len(models)} models.")

    for test_model in ("auto/best-free", "groq/openai/gpt-oss-120b", "gemini/gemini-3.5-flash", "openrouter/auto"):
        payload = json.dumps({
            "model": test_model,
            "messages": [{"role": "user", "content": "Reply with exactly OK."}],
            "max_tokens": 64,
        }).encode()
        creq = urllib.request.Request(
            f"http://127.0.0.1:{PORT}/v1/chat/completions",
            data=payload,
            headers={"Authorization": "Bearer " + api_key, "Content-Type": "application/json"},
            method="POST",
        )
        try:
            with urllib.request.urlopen(creq, timeout=20) as cresp:
                cbody = json.loads(cresp.read().decode())
                reply = cbody.get("choices", [{}])[0].get("message", {}).get("content", "").strip()
                print(f"  ✔ {test_model} -> {reply!r}")
        except urllib.error.HTTPError as e:
            print(f"  ⚠ {test_model} -> HTTP {e.code}: {e.read().decode()[:160]}")
finally:
    del api_key

## 6. Save Settings to Google Drive & Stop Server
Gracefully stops OmniRoute (flushing SQLite WAL to disk) and saves `/content/omniroute-data` to `/content/drive/MyDrive/OmniRoute/omniroute-backup.tar.gz`.

In [ ]:
import os
import shutil
import signal
import subprocess
import tarfile
from pathlib import Path

for proc_name in ("tunnel_process", "omniroute_process"):
    proc = globals().get(proc_name)
    if proc is not None and proc.poll() is None:
        try:
            os.killpg(proc.pid, signal.SIGTERM)
            proc.wait(timeout=25)
        except subprocess.TimeoutExpired:
            os.killpg(proc.pid, signal.SIGKILL)
            proc.wait()
        except Exception:
            pass

DATA = Path("/content/omniroute-data")
SAVE_DIR = Path("/content/drive/MyDrive/OmniRoute")
BACKUP = SAVE_DIR / "omniroute-backup.tar.gz"

if not (DATA / "storage.sqlite").is_file():
    raise RuntimeError("No storage.sqlite found in /content/omniroute-data. Start OmniRoute first.")

SAVE_DIR.mkdir(parents=True, exist_ok=True)
local_backup = Path("/content/omniroute-backup.tar.gz")
with tarfile.open(local_backup, "w:gz") as archive:
    archive.add(DATA, arcname="omniroute-data")

staged = SAVE_DIR / "omniroute-backup.uploading"
shutil.copyfile(local_backup, staged)
if BACKUP.exists():
    shutil.copyfile(BACKUP, SAVE_DIR / "omniroute-backup.previous.tar.gz")
staged.replace(BACKUP)
print(f"✅ OmniRoute stopped and settings backed up to {BACKUP} ({BACKUP.stat().st_size // 1024} KB).")

## Troubleshooting: Reset Password / Wipe Local State
Run this cell **only** if you want to wipe `/content/omniroute-data` in the current Colab runtime and set a new password from scratch (does not delete your Drive backup unless you run Step 6 afterwards).

In [ ]:
import os
import shutil
import signal

proc = globals().get("omniroute_process")
if proc is not None and proc.poll() is None:
    try:
        os.killpg(proc.pid, signal.SIGKILL)
        proc.wait()
    except Exception:
        pass

shutil.rmtree("/content/omniroute-data", ignore_errors=True)
print("✅ Local runtime data wiped. Rerun Step 1 to restore from Drive, or Step 3 to set a new password.")